In [ ]:
from pathlib import Path
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema

from setup_utils import create_minio_spark_session, DEV_LOCATION_ID_LIST

### Start the spark session

In [ ]:
spark = create_minio_spark_session()

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

### Initialize TEEHR Evaluation w/ read/write privileges

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

### Ingest FIRO test data into warehouse

#### Load core tables

In [ ]:
inputs_dir = Path(Path.cwd(), 'FIRO_data')

# load configurations (piece-wise) from parquet
configurations_path = Path(inputs_dir, 'configurations.parquet')
df = pd.read_parquet(configurations_path)
for _, row in df.iterrows():
    ev.configurations.add(
        teehr.Configuration(
            name=row['name'],
            timeseries_type=row['timeseries_type'],
            description=row['description']
        )
    )

# load units (piece_wise) from parquet
units_path = Path(inputs_dir, 'units.parquet')
df = pd.read_parquet(units_path)
for _, row in df.iterrows():
    ev.units.add(
        teehr.Unit(
            name=row['name'],
            long_name=row['long_name']
        )
    )

# load variables (piece-wise) from parquet
variables_path = Path(inputs_dir, 'variables.parquet')
df = pd.read_parquet(variables_path)
for _, row in df.iterrows():
    ev.variables.add(
        teehr.Variable(
            name=row["name"],
            long_name=row["long_name"],
        )
    )

# load locations from parquet
locations_path = Path(inputs_dir, 'locations.parquet')
ev.locations.load_spatial(locations_path)

# load crosswalks from parquet
location_crosswalks_path = Path(inputs_dir, 'location_crosswalks.parquet')
ev.location_crosswalks.load_parquet(location_crosswalks_path)

# load attributes from parquet
attributes_path = Path(inputs_dir, 'attributes.parquet')
ev.attributes.load_parquet(attributes_path)

# load location_attributes from parquet
location_attributes_path = Path(inputs_dir, 'location_attributes.parquet')
ev.location_attributes.load_parquet(location_attributes_path)

# load primary_timeseries from parquet
primary_timeseries_path = Path(inputs_dir, 'primary_timeseries.parquet')
ev.primary_timeseries.load_parquet(primary_timeseries_path)

# load secondary_timeseries from parquet
secondary_timeseries_path = Path(inputs_dir, 'secondary_timeseries.parquet')
ev.secondary_timeseries.load_parquet(secondary_timeseries_path)

### Kill spark

In [ ]:
spark.stop()